In [ ]:
papers_without_doi = df[df["DOI"].isna()]
print("Papers without DOI:")
print(papers_without_doi[["title", "DOI"]])

duplicate_doi = df[df.duplicated(subset="DOI", keep=False)]
same_doi_diff_title = duplicate_doi.groupby("DOI").filter(lambda x: x["title"].nunique() > 1)
print("\nPapers with the same DOI but different titles:")
print(same_doi_diff_title[["DOI", "title"]].sort_values("DOI"))

duplicate_title = df[df.duplicated(subset="title", keep=False)]
same_title_diff_doi = duplicate_title.groupby("title").filter(lambda x: x["DOI"].nunique() > 1)
print("\nPapers with the same title but different DOIs:")
print(same_title_diff_doi[["title", "DOI"]].sort_values("title"))

In [ ]:
import pandas as pd
import math
import ast

def _to_list(x):
    # None/NaN -> lista vazia
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return []
    # já iteráveis comuns
    if isinstance(x, (list, tuple, set)):
        return list(x)
    # string que parece lista: "['a','b']"
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                val = ast.literal_eval(s)
                return list(val) if isinstance(val, (list, tuple, set)) else [s]
            except Exception:
                return [s]
        # string simples -> trate como uma keyword única
        return [s]
    # fallback: trate o valor atômico como uma keyword
    return [x]

def _norm(kw):
    # normalização opcional (lowercase + strip)
    return str(kw).strip().lower()

def compare_keywords(group: pd.DataFrame) -> pd.Series:
    # normaliza cada célula da coluna keywords para lista
    lists_kw = [ _to_list(x) for x in group["keywords"] ]
    # aplica normalização por item e transforma em set (remove duplicatas)
    sets_kw = [ set(_norm(k) for k in lst if str(k).strip() != "") for lst in lists_kw ]
    # mantém apenas os não vazios
    sets_kw = [s for s in sets_kw if s]

    if len(sets_kw) < 2:
        return pd.Series({
            "num_papers": len(sets_kw),
            "intersecao": [],
            "tam_intersecao": 0,
            "disjuncao": [],
            "tam_disjuncao": 0
        })

    # interseção/união robustas
    intersecao = set.intersection(*sets_kw)
    uniao = set().union(*sets_kw)
    disjuncao = uniao - intersecao

    return pd.Series({
        "num_papers": len(sets_kw),
        "intersecao": sorted(intersecao),
        "tam_intersecao": len(intersecao),
        "disjuncao": sorted(disjuncao),
        "tam_disjuncao": len(disjuncao)
    })

# uso:
# keywords_summary = df.groupby("DOI", dropna=False).apply(compare_keywords).reset_index()
